In [1]:
# Route visualization notebook for GPS ping exports from this project
# If needed, install dependencies once:
# %pip install pandas folium matplotlib ipython

from pathlib import Path
import json
import math

import pandas as pd
import folium
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## 1) Choose a GPS export from recordedRuns
This cell lists JSON files in `recordedRuns/` and asks you to pick one by number.

In [2]:
data_dir = Path("recordedRuns")
json_files = sorted(
    data_dir.glob("*.json"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

if not json_files:
    raise FileNotFoundError(
        "No .json files found in recordedRuns/. "
        "Export or copy a run there, then rerun this cell."
    )

print("Available route files:")
for i, path in enumerate(json_files, start=1):
    print(f"{i}. {path.name}")

choice_raw = input("Choose a file number (press Enter for 1): ").strip()
choice = 1 if choice_raw == "" else int(choice_raw)

if choice < 1 or choice > len(json_files):
    raise ValueError(f"Pick a number between 1 and {len(json_files)}")

data_file = json_files[choice - 1]
print(f"Using file: {data_file}")

with data_file.open("r", encoding="utf-8") as f:
    payload = json.load(f)

pings = payload.get("pings", [])
if not pings:
    raise ValueError("The JSON file has no pings to visualize.")

df = pd.DataFrame(pings)
print(f"Loaded {len(df)} pings")
display(df.head())

Available route files:
1. car-drive-test-template.json
Using file: recordedRuns\car-drive-test-template.json
Loaded 8 pings


,timestampMs,timestampIso,latitude,longitude,accuracy,altitude,altitudeAccuracy,heading,speed
0,1783346400000,2026-07-06T14:00:00.000Z,35.149691,-78.988823,6.3,112.4,10.5,92,0.0
1,1783346410000,2026-07-06T14:00:10.000Z,35.148869,-78.988550,5.8,112.3,10.2,94,3.1
2,1783346420000,2026-07-06T14:00:20.000Z,35.148772,-78.988277,5.2,112.1,9.9,96,6.5
3,1783346430000,2026-07-06T14:00:30.000Z,35.149588,-78.984412,5.0,111.9,9.7,97,9.2
4,1783346440000,2026-07-06T14:00:40.000Z,35.149861,-78.984377,4.9,111.8,9.4,99,12.8


## 2) Clean, sort, and compute route distance
This cell parses timestamps, removes invalid rows, and calculates segment + cumulative distance in meters.

In [3]:
required = ["latitude", "longitude", "timestampIso"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns in export: {missing}")

route_df = df.copy()
route_df["timestamp"] = pd.to_datetime(route_df["timestampIso"], errors="coerce", utc=True)
route_df = route_df.dropna(subset=["latitude", "longitude", "timestamp"])
route_df = route_df.sort_values("timestamp").reset_index(drop=True)

if len(route_df) < 2:
    raise ValueError("Need at least 2 valid points to draw a route.")


def haversine_m(lat1, lon1, lat2, lon2):
    r = 6_371_000.0  # Earth radius in meters
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = (
        math.sin(dphi / 2) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    )
    return 2 * r * math.atan2(math.sqrt(a), math.sqrt(1 - a))

segment_m = [0.0]
for i in range(1, len(route_df)):
    prev = route_df.iloc[i - 1]
    curr = route_df.iloc[i]
    d = haversine_m(prev.latitude, prev.longitude, curr.latitude, curr.longitude)
    segment_m.append(d)

route_df["segment_m"] = segment_m
route_df["cumulative_km"] = route_df["segment_m"].cumsum() / 1000

trip_duration = route_df["timestamp"].iloc[-1] - route_df["timestamp"].iloc[0]
total_distance_km = route_df["cumulative_km"].iloc[-1]

print(f"Points used: {len(route_df)}")
print(f"Total route distance: {total_distance_km:.3f} km")
print(f"Trip duration: {trip_duration}")
display(route_df[["timestamp", "latitude", "longitude", "segment_m", "cumulative_km"]].head())

Points used: 8
Total route distance: 1.085 km
Trip duration: 0 days 00:01:10


,timestamp,latitude,longitude,segment_m,cumulative_km
0,2026-07-06 14:00:00+00:00,35.149691,-78.988823,0.000000,0.000000
1,2026-07-06 14:00:10+00:00,35.148869,-78.988550,94.735804,0.094736
2,2026-07-06 14:00:20+00:00,35.148772,-78.988277,27.078489,0.121814
3,2026-07-06 14:00:30+00:00,35.149588,-78.984412,362.919716,0.484734
4,2026-07-06 14:00:40+00:00,35.149861,-78.984377,30.436448,0.515170


## 3) Render the route on an interactive map
The polyline connects each geopoint in time order, with start/end markers and optional point markers.

In [4]:
coords = route_df[["latitude", "longitude"]].values.tolist()
center = [route_df["latitude"].mean(), route_df["longitude"].mean()]

route_map = folium.Map(location=center, zoom_start=14, tiles="CartoDB dark_matter")

folium.PolyLine(
    locations=coords,
    color="#00e0b8",
    weight=5,
    opacity=0.9,
    tooltip="Driven route",
).add_to(route_map)

start = coords[0]
end = coords[-1]

folium.Marker(
    location=start,
    popup="Start",
    tooltip="Start",
    icon=folium.Icon(color="green", icon="play"),
).add_to(route_map)

folium.Marker(
    location=end,
    popup="End",
    tooltip="End",
    icon=folium.Icon(color="red", icon="stop"),
).add_to(route_map)

# Add a marker every N points to keep large routes readable.
step = max(1, len(route_df) // 30)
for i in range(0, len(route_df), step):
    row = route_df.iloc[i]
    folium.CircleMarker(
        location=[row.latitude, row.longitude],
        radius=3,
        color="#8ce6ff",
        fill=True,
        fill_opacity=0.7,
        popup=f"#{i+1} | {row.timestamp}",
    ).add_to(route_map)

display(route_map)

## 4) Optional: save map to HTML
Use this if you want to share or open the route map outside the notebook.

In [5]:
output_map = data_file.with_suffix(".route-map.html")
route_map.save(output_map)
print(f"Saved interactive map to: {output_map}")

Saved interactive map to: recordedRuns\car-drive-test-template.route-map.html
